# OpenEnv ASR - Training Quickstart

This notebook demonstrates how to:
1. Set up the ASR environment
2. Train a Hybrid Actor-Critic Agent
3. Monitor training progress
4. Save and load checkpoints

## 1. Installation & Imports

In [ ]:
# Install package
!pip install -e /path/to/openenv-asr

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

from environment.gym_integration import ASREnvironmentHub, register_environments
from agents.hybrid_actor_critic_agent import HybridActorCriticAgent

## 2. Setup

In [ ]:
# Check device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Register environments
register_environments()
print("✅ Environments registered")

# Create environment (50 unique scenarios)
env = ASREnvironmentHub(num_scenarios=50, seed=42)
print(f"✅ Environment created with metadata:")
print(env.get_metadata())

In [ ]:
# Create agent
agent = HybridActorCriticAgent(
    input_dim=512,
    hidden_dim=256,
    num_actions=3,
    learning_rate=3e-4,
    device=device
)
print("✅ Agent created")
print(f"   Policy Network: {agent.policy_net}")
print(f"   Value Network: {agent.value_net}")

## 3. Training Loop (Mini Example)

In [ ]:
# Training configuration
num_episodes = 10
max_steps_per_episode = 50

# Tracking
episode_rewards = []
episode_lengths = []
avg_reward_window = deque(maxlen=5)

print(f"🚀 Starting training for {num_episodes} episodes")
print("-" * 50)

In [ ]:
for episode in range(num_episodes):
    # Reset environment
    obs = env.reset()
    episode_reward = 0
    episode_length = 0
    episode_rewards_list = []
    
    done = False
    step = 0
    
    while not done and step < max_steps_per_episode:
        # Agent selects action
        action_dict, confidence = agent.get_action({
            "file_tree": "",
            "current_file": "",
            "parse_tree": {},
            "test_results": {"passed": 0, "failed": 5},
            "last_output": "",
        })
        
        # Map action to index
        action_idx = 0 if action_dict["action"] == "read_file" else (
            1 if action_dict["action"] == "write_file" else 2
        )
        
        # Execute action
        obs, reward, done, info = env.step(action_idx)
        
        episode_reward += reward
        episode_length += 1
        episode_rewards_list.append(reward)
        step += 1
    
    # Update agent
    agent.update(episode_rewards_list)
    
    # Track statistics
    episode_rewards.append(episode_reward)
    episode_lengths.append(episode_length)
    avg_reward_window.append(episode_reward)
    
    # Print progress
    if (episode + 1) % 2 == 0:
        avg_reward = np.mean(list(avg_reward_window))
        print(f"Episode {episode + 1:3d}/{num_episodes} | "
              f"Reward: {episode_reward:7.1f} | "
              f"Avg (last 5): {avg_reward:7.1f} | "
              f"Length: {episode_length:3d}")

## 4. Visualize Training Progress

In [ ]:
# Plot results
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Episode rewards
axes[0].plot(episode_rewards, 'b-', alpha=0.6, label='Reward per episode')
axes[0].plot(np.convolve(episode_rewards, np.ones(3)/3, mode='valid'), 
             'r-', linewidth=2, label='Moving average (window=3)')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Training Progress - Rewards')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Episode lengths
axes[1].plot(episode_lengths, 'g-', alpha=0.6, label='Steps per episode')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Length')
axes[1].set_title('Training Progress - Episode Lengths')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_progress.png', dpi=100)
plt.show()

print("✅ Plot saved to training_progress.png")

## 5. Save Checkpoint

In [ ]:
import os
from pathlib import Path

checkpoint_dir = Path('./checkpoints')
checkpoint_dir.mkdir(exist_ok=True)

best_episode = np.argmax(episode_rewards)
best_reward = episode_rewards[best_episode]

checkpoint_path = checkpoint_dir / f'agent_ep{best_episode}_reward{best_reward:.0f}.pt'
agent.save(str(checkpoint_path))

print(f"✅ Checkpoint saved: {checkpoint_path}")
print(f"   Best episode: {best_episode + 1}")
print(f"   Best reward: {best_reward:.2f}")

## 6. Load and Evaluate Checkpoint

In [ ]:
# Create new agent
eval_agent = HybridActorCriticAgent(device=device)

# Load checkpoint
eval_agent.load(str(checkpoint_path))
print(f"✅ Loaded checkpoint: {checkpoint_path}")

# Set to eval mode
eval_agent.policy_net.eval()
eval_agent.value_net.eval()

In [ ]:
# Evaluate on new scenarios
num_eval_episodes = 5
eval_rewards = []

print(f"Evaluating on {num_eval_episodes} new scenarios...\n")

with torch.no_grad():
    for ep in range(num_eval_episodes):
        obs = env.reset()
        ep_reward = 0
        done = False
        step = 0
        
        while not done and step < max_steps_per_episode:
            action_dict, conf = eval_agent.get_action({
                "file_tree": "",
                "current_file": "",
                "parse_tree": {},
                "test_results": {"passed": 0, "failed": 5},
                "last_output": "",
            })
            
            action_idx = 0 if action_dict["action"] == "read_file" else (
                1 if action_dict["action"] == "write_file" else 2
            )
            
            obs, reward, done, info = env.step(action_idx)
            ep_reward += reward
            step += 1
        
        eval_rewards.append(ep_reward)
        print(f"Eval Episode {ep+1}: Reward = {ep_reward:.1f}")

print(f"\nAverage Eval Reward: {np.mean(eval_rewards):.2f}")
print(f"Std Dev: {np.std(eval_rewards):.2f}")

## 7. Inspect Scenario Information

In [ ]:
# Get generator statistics
scenario_stats = env.scenario_gen.get_statistics()

print("📊 Scenario Generation Statistics:")
print("-" * 50)
print(f"Total scenarios generated: {scenario_stats['total_scenarios_generated']}")
print(f"Number of bug templates: {scenario_stats['num_bug_templates']}")
print(f"Random seed: {scenario_stats['seed']}")
print(f"\nBug types available:")
for i, bug_type in enumerate(scenario_stats['bug_types'], 1):
    print(f"  {i}. {bug_type}")

## Summary

✅ Created and initialized ASR environment  
✅ Trained Hybrid Actor-Critic Agent for 10 episodes  
✅ Visualized training progress  
✅ Saved checkpoint  
✅ Evaluated on new scenarios  

Next steps:
1. Increase `num_episodes` and `num_scenarios` for real training
2. Monitor with Weights & Biases or TensorBoard
3. Try different hyperparameters
4. Evaluate on diverse bug patterns